# Core Visualizations

A runnable catalogue of the core result visualizations over the frozen
experiment database. Each figure renders inline and saves to disk, so the same
code produces the images and records the exact plotted values.

**Invariants**

1. Numbers come from `maestro.viz.queries` (canonical joins), never ad-hoc SQL.
   A helper query defined inline is flagged `# PROMOTE` as a candidate for
   `queries.py`, so the dashboard can reuse it.
2. Style comes from `maestro.viz.theme.apply_thesis_style()` and the palette
   helpers, the same code the dashboard uses. See
   `docs/visualization_design_guide.md`.
3. The database is pinned by sha256. Every figure reads that one file.
4. Each figure writes an SVG and a PNG, and records its plotted values into
   `figure_values.json`.

## Catalogue

| Visualization | Output file |
|---|---|
| Execution summary (values only) | (no figure) |
| Generated diagram vs ground truth | `diagram_vs_ground_truth` |
| Reliability funnel | `reliability_funnel` |
| Correctness with confidence intervals | `correctness_ci` |
| Pairwise strategy contrasts (Tukey HSD) | `strategy_contrasts_tukey` |
| Mean entity F1 per strategy (values only) | (no figure) |
| Correctness by input complexity | `correctness_by_complexity` |
| Error-mode profile | `error_mode_profile` |
| Error patterns across complexity | `error_by_complexity` |
| Correctness against cost | `correctness_vs_cost` |
| Model lever vs strategy lever | `model_cost_correctness` |
| Strategy x model robustness | `strategy_model_heatmap` |

Value-only sections produce numbers for `figure_values.json` without a figure.

## Setup: frozen DB, house style, figure/number sinks

In [ ]:
from __future__ import annotations

import hashlib
import json
import sqlite3
import sys
from pathlib import Path

# Bootstrap: if maestro is not installed on this kernel, add the repo src/
# to the path. No-op for a normal editable install (import succeeds first).
try:
    import maestro  # noqa: F401
except ModuleNotFoundError:
    _p = Path.cwd()
    while not (_p / "src" / "maestro").exists() and _p != _p.parent:
        _p = _p.parent
    sys.path.insert(0, str(_p / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats as st

from maestro.analysis import statistics as S
from maestro.viz import db as viz_db
from maestro.viz import queries as q
from maestro.viz import theme

REPO = Path.cwd()
while not (REPO / "out" / "maestro.db").exists() and REPO != REPO.parent:
    REPO = REPO.parent
DB_PATH = REPO / "out" / "maestro.db"
FIG_DIR = REPO / "out" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Pin to the frozen DB: fail loudly if the sha256 sidecar disagrees.
sha_file = REPO / "maestro.db.sha256"
if sha_file.exists():
    expected = sha_file.read_text().split()[0].strip()
    actual = hashlib.sha256(DB_PATH.read_bytes()).hexdigest()
    status = "OK" if actual == expected else "MISMATCH"
    print(f"DB sha256 {status}: {actual[:16]}...")
else:
    print("No sha256 sidecar found; skipping DB pin check.")

theme.apply_thesis_style()

# Strategies in design-guide order (light -> dark); controls handled separately.
STRATEGY_ORDER = ["single_agent", "sop_based", "crew_ai", "lang_graph"]

# Plotted values, dumped to figure_values.json at the end.
NUMBERS: dict = {}


def save_fig(fig, name: str, numbers: dict | None = None) -> None:
    fig.savefig(FIG_DIR / f"{name}.svg", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    if numbers is not None:
        NUMBERS[name] = numbers
    print(f"saved {name}.svg / .png")


def ordered(strategies):
    return [s for s in STRATEGY_ORDER if s in set(strategies)]


# Shared analysis frame, loaded once so every cell can use the canonical
# per-cell aggregation from analysis.statistics (S).
with viz_db.connect(DB_PATH) as _conn:
    _conn.row_factory = sqlite3.Row
    sdf = S.load_dataframe(_conn)
agg_itt = S.aggregate_experimental(sdf, S.INTENT_TO_TREAT)
agg_vo = S.aggregate_experimental(sdf, S.VALID_ONLY)

print("DB:", DB_PATH)
print("figures ->", FIG_DIR)
print("analysis rows:", len(sdf))

## Execution summary

Headline operational counts. Sets the empirical stage; no hypothesis.

In [ ]:
with viz_db.connect(DB_PATH) as conn:
    summary = q.overview_summary(conn)

NUMBERS["execution_summary"] = summary
summary

## Generated diagram vs ground truth

A qualitative side-by-side, so the format of a run-vs-ground-truth comparison
is clear before the metrics quantify it.

Example: single-agent / gpt-5.5 / tier 2 / bpmn_2_15 / run 5. Entity-ID F1
0.978, a representative near-ceiling valid run (all five repeats identical).
It differs from the truth in two ways: an extra directed edge
(ErrorBoundaryEvent_FraudDetected -> Activity_ManualCheck, which the truth
encodes as an attachment) and one gateway left without a node shape. Kept
unannotated on purpose: this shows the format, not the error analysis.

Rendered with mmdc, the same engine the scorer uses. If mmdc is absent (e.g.
this kernel), the two .mmd sources are written out for rendering elsewhere.

In [ ]:
import shutil
import subprocess
import tempfile

import matplotlib.image as mpimg

EX_RUN = "6e2c0ade-5892-4e65-8151-5de3f39dda81"  # single_agent/gpt-5.5/bpmn_2_15/run5
gt_code = (REPO / "data" / "15_bpmn_2_ground_truth.MMD").read_text()
with viz_db.connect(DB_PATH) as conn:
    gen_code = conn.execute(
        "SELECT output_diagram_code FROM run_results WHERE run_id = ?", (EX_RUN,)
    ).fetchone()[0]


def _render_png(code: str, out, scale: int = 3) -> bool:
    # Render Mermaid to PNG via mmdc (the scorer's engine). False if mmdc
    # is missing or the render fails, so the caller can fall back to source.
    if shutil.which("mmdc") is None:
        return False
    with tempfile.NamedTemporaryFile("w", suffix=".mmd", delete=False) as f:
        f.write(code)
        src = f.name
    try:
        subprocess.run(
            ["mmdc", "-i", src, "-o", str(out), "-b", "white", "-s", str(scale)],
            check=True,
            capture_output=True,
            timeout=90,
        )
        return out.exists()
    except Exception:
        return False
    finally:
        Path(src).unlink(missing_ok=True)


gt_png = FIG_DIR / "_ex_gt.png"
gen_png = FIG_DIR / "_ex_gen.png"
if _render_png(gt_code, gt_png) and _render_png(gen_code, gen_png):
    fig, axes = plt.subplots(1, 2, figsize=(13, 7))
    for ax, img, title in (
        (axes[0], gt_png, "Ground Truth"),
        (axes[1], gen_png, "Generated Visualization"),
    ):
        ax.imshow(mpimg.imread(img))
        ax.axis("off")
        # Anchor each image to the top of its panel. The two diagrams have
        # very different aspect ratios; without this the wide/short one is
        # centred vertically and sits well below the tall one.
        ax.set_anchor("N")
        ax.set_title(
            title, fontsize=14, fontweight="bold", color="#000000", pad=10, y=1.0
        )
    fig.subplots_adjust(top=0.92, wspace=0.05)
    save_fig(fig, "diagram_vs_ground_truth")
    plt.show()
else:
    (FIG_DIR / "example_bpmn_2_15_ground_truth.mmd").write_text(gt_code)
    (FIG_DIR / "example_bpmn_2_15_generated.mmd").write_text(gen_code)
    print("mmdc not on this kernel: wrote the two .mmd sources to", FIG_DIR)
    print("Run this cell where mmdc is installed to render the comparison.")

## Reliability funnel

Share of runs ending valid / run error / invalid, per strategy. The headline
reliability finding: single-agent almost never crashes but returns the most
unrenderable diagrams; the orchestrated strategies crash around 10 percent of
the time yet render more reliably when they survive.

Percent rather than counts, since every strategy ran the same 1,500 cells.

In [ ]:
with viz_db.connect(DB_PATH) as conn:
    outcomes = {r[0]: r[1:] for r in q.run_outcomes_by_strategy(conn)}

strats = ordered(outcomes)
labels = [theme.strategy_display_name(s) for s in strats]
totals = np.array([sum(outcomes[s]) for s in strats], dtype=float)
valids = np.array([outcomes[s][0] for s in strats], dtype=float)
invalids = np.array([outcomes[s][1] for s in strats], dtype=float)
errors = np.array([outcomes[s][2] for s in strats], dtype=float)

p_valid = 100.0 * valids / totals
p_error = 100.0 * errors / totals
p_invalid = 100.0 * invalids / totals

# Percent, not counts: every strategy ran the same 1500 cells, so the shares
# are what differ. Run error sits directly on the valid block so the two
# run-level outcomes stay adjacent and the unrenderable band reads as the
# top of the funnel.
fig, ax = plt.subplots(figsize=(9, 5))
bars = [
    (p_valid, np.zeros(len(strats)), [theme.strategy_color(s) for s in strats], None),
    (p_error, p_valid, theme.OUTCOME_ERROR_COLOR, "Run error"),
    (p_invalid, p_valid + p_error, theme.OUTCOME_INVALID_COLOR, "Invalid (no render)"),
]
for heights, bottoms, color, label in bars:
    ax.bar(labels, heights, bottom=bottoms, color=color, label=label, width=0.62)

# A segment thinner than this cannot hold its own label, so the label moves
# off the bar with a leader line. Direction matters: a label pushed above the
# stack would collide with the one already sitting in the top band, so a thin
# run-error band drops below its segment and a thin top band rises above.
MIN_INSIDE_PCT = 6.0
LEADER_OFFSET = 9.0


def annotate(i, n, share, mid, inside_color, drop=False):
    text = f"{n:,.0f} ({share:.1f}%)"
    if share >= MIN_INSIDE_PCT:
        ax.text(i, mid, text, ha="center", va="center", color=inside_color, fontsize=8)
        return
    y = mid - LEADER_OFFSET if drop else mid + LEADER_OFFSET
    ax.annotate(
        text,
        xy=(i, mid),
        xytext=(i, y),
        ha="center",
        va="top" if drop else "bottom",
        fontsize=8,
        color="#333333",
        arrowprops=dict(arrowstyle="-", color="#333333", linewidth=0.6),
    )


for i in range(len(strats)):
    annotate(i, valids[i], p_valid[i], p_valid[i] / 2, "#FFFFFF")
    annotate(
        i, errors[i], p_error[i], p_valid[i] + p_error[i] / 2, "#FFFFFF", drop=True
    )
    annotate(
        i,
        invalids[i],
        p_invalid[i],
        p_valid[i] + p_error[i] + p_invalid[i] / 2,
        "#333333",
    )

ax.set_ylim(0, 100)
ax.set_yticks([0, 25, 50, 75, 100])
ax.set_yticklabels([f"{v}%" for v in (0, 25, 50, 75, 100)])
ax.set_ylabel("Share of runs (%)")
ax.set_xlabel("Strategy")
ax.grid(axis="x", visible=False)

fig.tight_layout(rect=[0, 0.08, 1, 1])
leg = fig.legend(
    loc="lower center",
    ncol=2,
    frameon=False,
    fontsize=9,
    handlelength=1.3,
    handleheight=1.3,
    handletextpad=0.7,
    columnspacing=3.0,
)
plt.setp(leg.get_texts(), color="#333333")

save_fig(
    fig,
    "reliability_funnel",
    numbers={
        s: {
            "total": int(totals[i]),
            "valid": int(valids[i]),
            "invalid": int(invalids[i]),
            "error": int(errors[i]),
            "valid_pct": round(p_valid[i], 1),
            "invalid_pct": round(p_invalid[i], 1),
            "error_pct": round(p_error[i], 1),
        }
        for i, s in enumerate(strats)
    },
)
plt.show()

## Mean entity F1 per strategy (values only)

Per-strategy mean `entity_id_f1` under the two scoring rules, reproducible
from the frozen DB. These are raw grand means over the runs, distinct from
the per-cell aggregated means used for the inferential tests (they differ by
at most 0.001).

In [ ]:
STRAT4 = tuple(STRATEGY_ORDER)
PH4 = ",".join(["?"] * len(STRAT4))

# Mean entity_id_f1 under both scoring conventions, from the shared query so
# this values-only cell and the corresponding table read one source.
with viz_db.connect(DB_PATH) as conn:
    _by = {
        s: (vo, itt)
        for s, vo, itt in q.mean_entity_id_f1_by_strategy_by_convention(conn)
    }

NUMBERS["entity_f1_by_strategy"] = {
    s: {"valid_only": round(_by[s][0], 3), "intent_to_treat": round(_by[s][1], 3)}
    for s in ordered(_by)
}
NUMBERS["entity_f1_by_strategy"]

## Correctness with confidence intervals

The inferential message, not just the point estimate. Per-cell means
(strategy x model x example, repeats averaged), then the strategy mean with a
95 percent confidence interval. Overlapping intervals are the visual statement
that the strategies do not differ. Shown under both scoring conventions; both
overlap.

In [ ]:
with viz_db.connect(DB_PATH) as conn:
    dfa = pd.read_sql_query(
        f"SELECT c.strategy, c.model, c.example_id, "
        f"COALESCE(m.parses_valid,0) AS valid, m.entity_id_f1 AS f1 "
        f"FROM run_configs c LEFT JOIN metric_results m ON c.run_id=m.run_id "
        f"WHERE c.strategy IN ({PH4})",
        conn,
        params=STRAT4,
    )

dfa["f1_valid"] = np.where(dfa["valid"] == 1, dfa["f1"], np.nan)
dfa["f1_itt"] = np.where(dfa["valid"] == 1, dfa["f1"], 0.0)


def strat_ci(col):
    # per-cell means, then strategy mean +/- 95% CI across cells (t-based).
    cells = dfa.groupby(["strategy", "model", "example_id"])[col].mean().reset_index()
    out = {}
    for s in ordered(STRATEGY_ORDER):
        v = cells[cells.strategy == s][col].dropna().values
        m = float(v.mean())
        h = float(st.sem(v) * st.t.ppf(0.975, len(v) - 1))
        out[s] = (m, h)
    return out


ci_valid = strat_ci("f1_valid")
ci_itt = strat_ci("f1_itt")
strats = ordered(STRATEGY_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
panels = [
    (axes[0], ci_valid, "Valid diagrams only", (0.90, 1.0)),
    (axes[1], ci_itt, "Every run (failures scored 0)", (0.75, 0.95)),
]
for ax, ci, title, ylim in panels:
    xs = list(range(len(strats)))
    ax.errorbar(
        xs,
        [ci[s][0] for s in strats],
        yerr=[ci[s][1] for s in strats],
        fmt="none",
        ecolor="#333333",
        capsize=4,
        zorder=2,
    )
    for i, s in enumerate(strats):
        ax.plot(
            i, ci[s][0], "o", color=theme.strategy_color(s), markersize=10, zorder=3
        )
    ax.set_xticks(xs)
    ax.set_xticklabels(
        [theme.strategy_display_name(s) for s in strats], rotation=30, ha="right"
    )
    ax.set_ylim(*ylim)
    ax.set_title(title, fontsize=10)
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("Entity-ID F1 (mean, 95% CI)")
fig.tight_layout()

save_fig(
    fig,
    "correctness_ci",
    numbers={
        "valid_only": {
            s: {
                "mean": round(ci_valid[s][0], 3),
                "ci95_halfwidth": round(ci_valid[s][1], 3),
            }
            for s in strats
        },
        "intent_to_treat": {
            s: {
                "mean": round(ci_itt[s][0], 3),
                "ci95_halfwidth": round(ci_itt[s][1], 3),
            }
            for s in strats
        },
    },
)
plt.show()

## Pairwise strategy contrasts (Tukey HSD)

The per-pair view behind the omnibus null: each strategy difference in mean
entity-ID F1 with its Tukey simultaneous 95 percent CI, sorted by adjusted
p-value. Every interval crosses zero, so no pair separates. Intent-to-treat
convention, matching the ANOVA.

In [ ]:
tukey = S.posthoc_strategy(sdf, S.INTENT_TO_TREAT)

# Label each contrast "group2 - group1" while plotting the canonical
# group1 - group2 difference statsmodels returns, so a leftward point means
# the second-named strategy scored lower. Sorted by adjusted p ascending, so
# the least-indistinguishable pair sits on top.
rows = sorted(tukey["comparisons"], key=lambda c: c["p_adj"])
labels = [
    f"{theme.strategy_display_name(c['group2'])} - "
    f"{theme.strategy_display_name(c['group1'])}"
    for c in rows
]
diffs = [c["meandiff"] for c in rows]
lower = [c["meandiff"] - c["lower"] for c in rows]
upper = [c["upper"] - c["meandiff"] for c in rows]

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ys = list(range(len(rows)))[::-1]
ax.axvline(0.0, color="#333333", linestyle="--", linewidth=0.9, zorder=1)
ax.errorbar(
    diffs, ys, xerr=[lower, upper], fmt="none", ecolor="#333333", capsize=4, zorder=2
)
ax.scatter(diffs, ys, s=90, color=theme.strategy_color("crew_ai"), zorder=3)
for y, c in zip(ys, rows):
    ax.annotate(
        f"p = {c['p_adj']:.2f}",
        xy=(c["upper"], y),
        xytext=(8, 0),
        textcoords="offset points",
        va="center",
        fontsize=9,
        color="#333333",
    )

ax.set_yticks(ys)
ax.set_yticklabels(labels)
ax.set_xlabel("Difference in mean entity-ID F1 (Tukey HSD, 95% CI)")
ax.grid(axis="y", visible=False)
ax.margins(x=0.18)

fig.tight_layout()
# Record the canonical statsmodels contrast (group1 - group2) with its own
# sign and CI, keyed unambiguously by that pair. The plot keeps the thesis
# label orientation; the values file stays free of it, so a reader never has
# to reconcile a label against an opposite sign.
save_fig(
    fig,
    "strategy_contrasts_tukey",
    numbers={
        f"{c['group1']} - {c['group2']}": {
            "meandiff": round(c["meandiff"], 4),
            "ci_low": round(c["lower"], 4),
            "ci_high": round(c["upper"], 4),
            "p_adj": round(c["p_adj"], 4),
            "reject": c["reject"],
        }
        for c in rows
    },
)
plt.show()

## Correctness by input complexity

`entity_id_f1` by strategy x tier. Survivorship caveat: valid-only means at
tier 3 are a biased sample.

In [ ]:
from matplotlib.patches import Patch

# Use the canonical per-cell aggregation from analysis.statistics so the
# figure matches the two-way ANOVA exactly (same convention, same grain).
agg_itt = S.aggregate_experimental(sdf, S.INTENT_TO_TREAT)
agg_vo = S.aggregate_experimental(sdf, S.VALID_ONLY)
tiers = [1, 2, 3]
# Bare tier labels; the strata definitions live in the figure note.
TIER_LABELS = {1: "Tier 1", 2: "Tier 2", 3: "Tier 3"}


def cell_ci(agg, s, t):
    # Mean and 95% CI half-width over the per-cell means in one strategy x tier.
    v = agg[(agg.strategy == s) & (agg.tier == t)]["entity_id_f1"].dropna().values
    if len(v) < 2:
        return (float(v.mean()) if len(v) else float("nan")), 0.0
    return float(v.mean()), float(st.sem(v) * st.t.ppf(0.975, len(v) - 1))


fig, axes = plt.subplots(1, 2, figsize=(10, 5.0), sharey=True)
for ax, agg, title in (
    (axes[0], agg_itt, "Every run (failures scored 0)"),
    (axes[1], agg_vo, "Valid diagrams only"),
):
    for s in ordered(list(agg["strategy"].unique())):
        pts = [cell_ci(agg, s, t) for t in tiers]
        ax.errorbar(
            tiers,
            [p[0] for p in pts],
            yerr=[p[1] for p in pts],
            marker="o",
            linewidth=2,
            capsize=3,
            markersize=7,
            color=theme.strategy_color(s),
            label=theme.strategy_display_name(s),
        )
    ax.set_xticks(tiers)
    ax.set_xticklabels([TIER_LABELS[t] for t in tiers], fontsize=9)
    # Colours set explicitly so the figure survives any ambient style.
    ax.set_xlabel("Input complexity", color="#000000")
    ax.set_title(title, fontsize=11, fontweight="bold", color="#000000")
    ax.set_xlim(0.6, 3.4)
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("Entity-identification F1 (mean, 95% CI)", color="#000000")

# One shared strategy key along the bottom, rather than a legend sitting
# inside a panel where it collides with the data. Square colour swatches
# read as a categorical key; repeating the line-and-errorbar marker adds
# nothing, since the encoding being explained is colour.
key = [
    Patch(facecolor=theme.strategy_color(s), label=theme.strategy_display_name(s))
    for s in ordered(STRATEGY_ORDER)
]
fig.tight_layout(rect=[0, 0.10, 1, 1])
leg = fig.legend(
    handles=key,
    loc="lower center",
    ncol=4,
    frameon=False,
    fontsize=10,
    handlelength=1.3,
    handleheight=1.3,
    handletextpad=0.7,
    columnspacing=6.0,
)
plt.setp(leg.get_texts(), color="#333333")

save_fig(
    fig,
    "correctness_by_complexity",
    numbers={
        "intent_to_treat": {
            s: {
                t: {
                    "mean": round(cell_ci(agg_itt, s, t)[0], 3),
                    "ci95": round(cell_ci(agg_itt, s, t)[1], 3),
                }
                for t in tiers
            }
            for s in ordered(STRATEGY_ORDER)
        },
        "valid_only": {
            s: {
                t: {
                    "mean": round(cell_ci(agg_vo, s, t)[0], 3),
                    "ci95": round(cell_ci(agg_vo, s, t)[1], 3),
                }
                for t in tiers
            }
            for s in ordered(STRATEGY_ORDER)
        },
    },
)
plt.show()

## Error-mode profile

Diagram-specific taxonomy: element type x error mode. Distinct signatures:
single-agent drops relationships; lang_graph over-generates entities.
`duplicate_*` is all zeros (one axis empty).

In [ ]:
from matplotlib.patches import Patch

ENT = [
    ("missing_entities", "Missing"),
    ("extra_entities", "Extra"),
    ("false_entities", "False"),
]
REL = [
    ("missing_relationships", "Missing"),
    ("extra_relationships", "Extra"),
    ("false_relationships", "False"),
]
ALLC = ENT + REL

# Mean error count per valid diagram, from the shared query so the figure
# and the corresponding table read one source.
with viz_db.connect(DB_PATH) as conn:
    rates = q.taxonomy_rates_per_valid_diagram(conn, tuple(col for col, _ in ALLC))

strats = ordered(STRATEGY_ORDER)
w = 0.2
fig, axes = plt.subplots(1, 2, figsize=(10, 5.0), sharey=True)
for ax, group, title in ((axes[0], ENT, "Entities"), (axes[1], REL, "Relationships")):
    x = np.arange(len(group))
    for j, s in enumerate(strats):
        ax.bar(
            x + (j - 1.5) * w,
            [rates[s][col] for col, _ in group],
            w,
            color=theme.strategy_color(s),
            label=theme.strategy_display_name(s),
        )
    ax.set_xticks(x)
    ax.set_xticklabels([lab for _, lab in group], fontsize=9)
    ax.set_xlabel("Error mode", color="#000000")
    ax.set_title(title, fontsize=11, fontweight="bold", color="#000000")
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("Mean errors per valid diagram", color="#000000")

key = [
    Patch(facecolor=theme.strategy_color(s), label=theme.strategy_display_name(s))
    for s in strats
]
fig.tight_layout(rect=[0, 0.10, 1, 1])
leg = fig.legend(
    handles=key,
    loc="lower center",
    ncol=4,
    frameon=False,
    fontsize=10,
    handlelength=1.3,
    handleheight=1.3,
    handletextpad=0.7,
    columnspacing=6.0,
)
plt.setp(leg.get_texts(), color="#333333")

save_fig(
    fig,
    "error_mode_profile",
    numbers={s: {col: round(rates[s][col], 3) for col, _ in ALLC} for s in strats},
)
plt.show()

## Error patterns across complexity

The two error modes that carry a real signal across tiers, normalised by
truth size. Normalisation matters: raw per-diagram counts triple from tier 1
to tier 3 mostly because the diagrams themselves triple in size (6.2 -> 32.8
truth relationships), which would otherwise be mistaken for degradation.

The other four modes (false entities, false relationships, extra
relationships, and entity omission) move non-monotonically at rates near
0.01 and are treated as noise, so they are not plotted.

In [ ]:
# PROMOTE: size-normalised error rates by strategy x tier, valid diagrams only.
with viz_db.connect(DB_PATH) as conn:
    dfe = pd.read_sql_query(
        "SELECT c.strategy, c.tier, "
        "1.0*m.missing_relationships/NULLIF(m.relationships_in_truth,0) AS miss_r, "
        "1.0*m.extra_entities/NULLIF(m.entities_in_truth,0) AS extra_e "
        "FROM run_configs c JOIN metric_results m ON c.run_id = m.run_id "
        f"WHERE m.parses_valid = 1 AND c.strategy IN ({PH4})",
        conn,
        params=STRAT4,
    )


def rate_ci(col, s, t):
    v = dfe[(dfe.strategy == s) & (dfe.tier == t)][col].dropna().values
    if len(v) < 2:
        return (float(v.mean()) if len(v) else float("nan")), 0.0
    return float(v.mean()), float(st.sem(v) * st.t.ppf(0.975, len(v) - 1))


# Single panel: only the relationship-omission trend survives its own
# confidence intervals. The extra-entity rate falls with complexity for every
# strategy, and LangGraph's tier-3 spike carries a CI of +-0.0485 spanning
# zero (outlier-driven), so it is treated as unreliable, not
# plotted as if it were a pattern.
PANELS = [
    ("miss_r", "Missing relationships", "Missing relationships per truth relationship")
]
fig, ax = plt.subplots(figsize=(7.5, 5.0))
for col, title, ylab in PANELS:
    for s in ordered(STRATEGY_ORDER):
        pts = [rate_ci(col, s, t) for t in tiers]
        ax.errorbar(
            tiers,
            [p[0] for p in pts],
            yerr=[p[1] for p in pts],
            marker="o",
            linewidth=2,
            capsize=3,
            markersize=7,
            color=theme.strategy_color(s),
            label=theme.strategy_display_name(s),
        )
    ax.set_xticks(tiers)
    ax.set_xticklabels([TIER_LABELS[t] for t in tiers], fontsize=9)
    ax.set_xlabel("Input complexity", color="#000000")
    ax.set_ylabel(ylab, color="#000000")
    ax.set_xlim(0.6, 3.4)
    ax.grid(axis="x", visible=False)

key = [
    Patch(facecolor=theme.strategy_color(s), label=theme.strategy_display_name(s))
    for s in ordered(STRATEGY_ORDER)
]
fig.tight_layout(rect=[0, 0.10, 1, 1])
leg = fig.legend(
    handles=key,
    loc="lower center",
    ncol=4,
    frameon=False,
    fontsize=10,
    handlelength=1.3,
    handleheight=1.3,
    handletextpad=0.7,
    columnspacing=6.0,
)
plt.setp(leg.get_texts(), color="#333333")

save_fig(
    fig,
    "error_by_complexity",
    numbers={
        col: {
            s: {
                t: {
                    "mean": round(rate_ci(col, s, t)[0], 4),
                    "ci95": round(rate_ci(col, s, t)[1], 4),
                }
                for t in tiers
            }
            for s in ordered(STRATEGY_ORDER)
        }
        for col, _, _ in PANELS
    },
)
plt.show()

## Correctness against cost

Entity-ID F1 vs cost per run. Colour = strategy, marker = tier. Cost on a log
axis (spans orders of magnitude, per the design guide).

In [ ]:
from matplotlib.lines import Line2D

# Cost over ALL runs, not just successful ones: a failed run still spends
# tokens, and the user pays for it. Correctness is intent-to-treat so both
# axes account for failures the same way.
with viz_db.connect(DB_PATH) as conn:
    cost = pd.read_sql_query(
        "SELECT c.strategy, c.tier, AVG(r.cost_usd) AS usd, "
        "AVG(r.prompt_tokens + r.completion_tokens) AS tok, "
        "AVG(r.duration_ms)/1000.0 AS sec "
        "FROM run_configs c JOIN run_results r ON c.run_id = r.run_id "
        f"WHERE c.strategy IN ({PH4}) GROUP BY c.strategy, c.tier",
        conn,
        params=STRAT4,
    )

f1t = agg_itt.groupby(["strategy", "tier"])["entity_id_f1"].mean().reset_index()
m = cost.merge(f1t, on=["strategy", "tier"])
m["ratio"] = m["entity_id_f1"] / m["usd"]
TIER_MARK = {1: "o", 2: "s", 3: "^"}

fig, axes = plt.subplots(1, 2, figsize=(11, 5.0))

ax = axes[0]
for s in ordered(STRATEGY_ORDER):
    d = m[m.strategy == s].sort_values("tier")
    ax.plot(
        d.usd,
        d.entity_id_f1,
        "-",
        color=theme.strategy_color(s),
        alpha=0.45,
        linewidth=1.2,
        zorder=1,
    )
    for _, row in d.iterrows():
        ax.scatter(
            row.usd,
            row.entity_id_f1,
            marker=TIER_MARK[int(row.tier)],
            s=80,
            color=theme.strategy_color(s),
            zorder=3,
        )
ax.set_xlabel("Mean cost per run (USD)", color="#000000")
ax.set_ylabel("Entity-identification F1 (mean)", color="#000000")
ax.set_title(
    "Correctness against cost", fontsize=11, fontweight="bold", color="#000000"
)
mk = [
    Line2D(
        [],
        [],
        marker=TIER_MARK[t],
        color="#333333",
        linestyle="none",
        markersize=7,
        label=f"Tier {t}",
    )
    for t in tiers
]
tl = ax.legend(handles=mk, frameon=False, fontsize=8, loc="upper right")
plt.setp(tl.get_texts(), color="#333333")

ax2 = axes[1]
x = np.arange(len(tiers))
w = 0.2
for j, s in enumerate(ordered(STRATEGY_ORDER)):
    vals = [float(m[(m.strategy == s) & (m.tier == t)]["ratio"].iloc[0]) for t in tiers]
    ax2.bar(x + (j - 1.5) * w, vals, w, color=theme.strategy_color(s))
ax2.set_xticks(x)
ax2.set_xticklabels([TIER_LABELS[t] for t in tiers], fontsize=9)
ax2.set_xlabel("Input complexity", color="#000000")
ax2.set_ylabel("Entity-identification F1 per USD", color="#000000")
ax2.set_title(
    "Correctness per unit of cost", fontsize=11, fontweight="bold", color="#000000"
)
ax2.grid(axis="x", visible=False)

key = [
    Patch(facecolor=theme.strategy_color(s), label=theme.strategy_display_name(s))
    for s in ordered(STRATEGY_ORDER)
]
fig.tight_layout(rect=[0, 0.10, 1, 1])
leg = fig.legend(
    handles=key,
    loc="lower center",
    ncol=4,
    frameon=False,
    fontsize=10,
    handlelength=1.3,
    handleheight=1.3,
    handletextpad=0.7,
    columnspacing=6.0,
)
plt.setp(leg.get_texts(), color="#333333")

save_fig(
    fig,
    "correctness_vs_cost",
    numbers={
        f"{r.strategy}_T{int(r.tier)}": {
            "usd_per_run": round(r.usd, 4),
            "tokens_per_run": round(r.tok, 0),
            "sec_per_run": round(r.sec, 1),
            "f1": round(r.entity_id_f1, 3),
            "f1_per_usd": round(r.ratio, 1),
        }
        for r in m.itertuples()
    },
)
plt.show()

## Model lever vs strategy lever

Model pricing varies far more than orchestration does: 50x across models
against 3x across strategies. The decisive contrast is that paying more for a
model *does* buy correctness (F1 spans 0.616 to 0.992) whereas paying more for
orchestration does not (0.818 to 0.858, not significant).

Scope: the efficiency angle only. Whether the strategy findings replicate
across models is the next section.

In [ ]:
from maestro.viz.theme import _MODEL_TO_PROVIDER_SLOT as M2PS

# Short labels keep the version, which identifies the exact model, but drop
# the vendor (the provider colour and legend already carry it) and the build
# date stamp. Mistral Small has no semantic version in its id, only a build
# code, so it is left unversioned rather than shown as '2603'.
SHORT = {
    "claude-haiku-4-5-20251001": "haiku 4.5",
    "claude-opus-4-8": "opus 4.8",
    "gpt-5.4-mini-2026-03-17": "gpt-5.4 mini",
    "gpt-5.5-2026-04-23": "gpt-5.5",
    "mistral-small-2603": "mistral small",
    "mistral-medium-3-5": "mistral medium 3.5",
    "gemini-3.1-flash-lite": "3.1 flash-lite",
    "gemini-3.5-flash": "3.5 flash",
    "deepseek-v4-flash": "ds v4 flash",
    "deepseek-v4-pro": "ds v4 pro",
}

with viz_db.connect(DB_PATH) as conn:
    mcost = pd.read_sql_query(
        "SELECT c.model, AVG(r.cost_usd) AS usd "
        "FROM run_configs c JOIN run_results r ON c.run_id = r.run_id "
        f"WHERE c.strategy IN ({PH4}) GROUP BY c.model",
        conn,
        params=STRAT4,
    )
mf1 = agg_itt.groupby("model")["entity_id_f1"].mean().reset_index()
mm = mcost.merge(mf1, on="model")
mm["slot"] = [M2PS[x][1] for x in mm.model]

# Label placement: default above the marker. A few are moved below or
# left-aligned where markers sit close together or near the axis.
# The widened x-limits leave room for the outermost labels, so everything is
# centred above its marker except the three moved below to avoid collisions.
LBL = {
    "mistral medium 3.5": (0, -19, "center"),
    "gpt-5.5": (0, -19, "center"),
    "ds v4 pro": (0, -19, "center"),
}

fig, ax = plt.subplots(figsize=(9, 5.4))
for r in mm.itertuples():
    ax.scatter(
        r.usd,
        r.entity_id_f1,
        marker="D" if r.slot else "o",
        s=95,
        color=theme.model_color(r.model),
        zorder=3,
    )
    dx, dy, ha = LBL.get(SHORT[r.model], (0, 12, "center"))
    ax.annotate(
        SHORT[r.model],
        (r.usd, r.entity_id_f1),
        textcoords="offset points",
        xytext=(dx, dy),
        ha=ha,
        fontsize=9,
        color="#333333",
    )
ax.set_xscale("log")
# Extra horizontal room so the outermost labels clear the axes.
ax.set_xlim(mm.usd.min() * 0.55, mm.usd.max() * 1.9)
ax.set_xlabel("Mean cost per run in USD (log)", color="#000000")
ax.set_ylabel("Entity-identification F1 (mean)", color="#000000")
ax.set_ylim(0.55, 1.06)

slotkey = [
    Line2D(
        [],
        [],
        marker="o",
        color="#333333",
        linestyle="none",
        markersize=7,
        label="Efficiency model",
    ),
    Line2D(
        [],
        [],
        marker="D",
        color="#333333",
        linestyle="none",
        markersize=7,
        label="Frontier model",
    ),
]
sk = ax.legend(handles=slotkey, frameon=False, fontsize=8, loc="lower right")
plt.setp(sk.get_texts(), color="#333333")

PROVIDERS = ["Claude", "ChatGPT", "Mistral", "Gemini", "DeepSeek"]
pkey = [Patch(facecolor=theme.PROVIDER_TIERS[p][1], label=p) for p in PROVIDERS]
fig.tight_layout(rect=[0, 0.09, 1, 1])
leg = fig.legend(
    handles=pkey,
    loc="lower center",
    ncol=5,
    frameon=False,
    fontsize=9,
    handlelength=1.3,
    handleheight=1.3,
    handletextpad=0.6,
    columnspacing=3.0,
)
plt.setp(leg.get_texts(), color="#333333")

save_fig(
    fig,
    "model_cost_correctness",
    numbers={
        r.model: {
            "usd_per_run": round(r.usd, 4),
            "f1": round(r.entity_id_f1, 3),
            "f1_per_usd": round(r.entity_id_f1 / r.usd, 1),
            "tier": "frontier" if r.slot else "efficiency",
        }
        for r in mm.itertuples()
    },
)
plt.show()

## Strategy x model robustness

`entity_id_f1` heatmap, strategy x model. Model is a replication dimension, so
this is a robustness check: does the strategy pattern hold across models.
vmin/vmax pinned 0-1 per the design guide.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Intent-to-treat, per-cell aggregated: same convention and grain as the
# strategy x model ANOVA, so figure and test agree. Models ordered by their
# own mean, so the column gradient is readable.
piv_m = agg_itt.pivot_table(
    index="strategy", columns="model", values="entity_id_f1", aggfunc="mean"
)
models = list(piv_m.mean(axis=0).sort_values().index)
strats = ordered(list(piv_m.index))
M = np.array([[piv_m.loc[s, m] for m in models] for s in strats])

# Sequential ramp built from the tier amber family rather than YlOrRd. The
# tier palette is never used as a colour encoding elsewhere,
# and the strategy pinks are unusable here because strategy is already the
# row dimension. vmin/vmax stay at 0/1: the data occupy 0.56-1.00, so the
# band looks flat, which is the honest picture given the null result.
# Compressing the scale would manufacture contrast the statistics deny.
TIER_SEQ = LinearSegmentedColormap.from_list(
    "tier_sequential", ["#FFFFFF", "#FAC775", "#BA7517", "#633806"]
)

fig, ax = plt.subplots(figsize=(11, 4.2))
im = ax.imshow(M, cmap=TIER_SEQ, vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(models)))
ax.set_xticklabels(
    [SHORT.get(m, m) for m in models], rotation=30, ha="right", fontsize=9
)
ax.set_yticks(range(len(strats)))
ax.set_yticklabels([theme.strategy_display_name(s) for s in strats], fontsize=9)
# Heatmap exception in the design guide: full box frame, no gridlines.
for spine in ax.spines.values():
    spine.set_visible(True)
ax.grid(False)


def _wcag_luminance(rgb):
    # WCAG relative luminance: each sRGB channel must be gamma-linearised
    # first. A plain weighted sum of sRGB values is not the same quantity and
    # puts the black/white crossover in the wrong place.
    def lin(c):
        return c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4

    r, g, b = (lin(c) for c in rgb[:3])
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def _contrast(fg, bg):
    hi, lo = sorted((_wcag_luminance(fg), _wcag_luminance(bg)), reverse=True)
    return (hi + 0.05) / (lo + 0.05)


def _txt_colour(v):
    # Pick whichever of black/white gives the higher true WCAG ratio against
    # this cell. Pure black rather than the house #333333: on mid amber there
    # is a band where neither white nor #333333 reaches AA, and only black
    # does. Verified below: every cell clears 4.5:1 for normal text.
    bg = TIER_SEQ(v)[:3]
    return (
        "#000000" if _contrast((0, 0, 0), bg) >= _contrast((1, 1, 1), bg) else "#FFFFFF"
    )


for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        if not np.isnan(M[i, j]):
            ax.text(
                j,
                i,
                f"{M[i, j]:.2f}",
                ha="center",
                va="center",
                color=_txt_colour(M[i, j]),
                fontsize=8,
            )
cbar = fig.colorbar(im, ax=ax, fraction=0.030, pad=0.02)
cbar.set_label("Entity-identification F1", color="#000000")
fig.tight_layout()

# Accessibility gate: every cell label must clear WCAG AA for normal text.
_ratios = []
for _v in M.flatten():
    if not np.isnan(_v):
        _fg = (0, 0, 0) if _txt_colour(_v) == "#000000" else (1, 1, 1)
        _ratios.append(_contrast(_fg, TIER_SEQ(_v)[:3]))
print(
    f"WCAG contrast: min {min(_ratios):.2f}:1, max {max(_ratios):.2f}:1 "
    f"over {len(_ratios)} cells"
)
assert min(_ratios) >= 4.5, f"cell label below WCAG AA: {min(_ratios):.2f}:1"

save_fig(
    fig,
    "strategy_model_heatmap",
    numbers={s: {m: round(float(piv_m.loc[s, m]), 3) for m in models} for s in strats},
)
plt.show()

## Emit figure_values.json

Single source of truth for every plotted value.

In [ ]:
out = FIG_DIR / "figure_values.json"
out.write_text(json.dumps(NUMBERS, indent=2))
print("wrote", out)
print(json.dumps(NUMBERS, indent=2)[:1500])